## CS310 Natural Language Processing
## Lab 7: Decoding with LLMs

In this lab, we will practice some decoding strategies for LLM inference.

Download the model weights here: https://huggingface.co/Qwen/Qwen3-0.6B. Or you can load the model with path "Qwen/Qwen3-0.6B" in `transformers`.

In [2]:
import torch
import torch.nn.functional as F
import random
import numpy as np
import time

## T1. Explore the Model

First, we will explore the Qwen3-0.6B model and get familiar with some of the APIs of `transformers`.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache

MODEL_PATH = "/Users/xy/models/Qwen3-0.6B/"

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# Evaluation mode
model = model.eval()

print('vocab size:', tokenizer.vocab_size)
print(f'special token {tokenizer.pad_token}:', tokenizer.pad_token_id)
print(f'special token id {tokenizer.pad_token_id}:', tokenizer.pad_token_id)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

vocab size: 151643
special token <|endoftext|>: 151643
special token id 151643: 151643


The tokenizer can return the token IDs and the attention mask that indicates which tokens are padding tokens (`1` for real tokens, `0` for padding tokens).

Since we only have one sentence in the "batch", there is no padding used, and thus no `0` in the attention mask.

In [4]:
input_text = '学而时习之，不亦说乎！'
input_encoded = tokenizer(input_text, return_tensors="pt")

print('input ids:', input_encoded['input_ids'])
print('input attention mask:', input_encoded['attention_mask'])

# Map token ids back to tokens
input_ids = input_encoded['input_ids'][0]

input_tokens = tokenizer.convert_ids_to_tokens(input_encoded['input_ids'][0])
print('input tokens:', input_tokens)

input_token_strings = [tokenizer.convert_tokens_to_string([tok]) for tok in input_tokens]
print('input token strings:', input_token_strings)

input ids: tensor([[ 47764,  68536,  13343,  99347,  53930,   3837,  16530, 103972,  36587,
          99587,   6313]])
input attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
input tokens: ['åŃ¦', 'èĢĮ', 'æĹ¶', 'ä¹ł', 'ä¹ĭ', 'ï¼Į', 'ä¸į', 'äº¦', 'è¯´', 'ä¹İ', 'ï¼ģ']
input token strings: ['学', '而', '时', '习', '之', '，', '不', '亦', '说', '乎', '！']


It's easy to directly use the `generate` method to generate some sentences.

The `generate` method takes multiple arguments as input. As a Python syntactic sugar, you can use the `**` operator to unpack a dictionary and pass the key-value pairs as individual keyword arguments.

For instance, `model.generate(**input_encoded)` is equivalent to `model.generate(input_ids=input_encoded['input_ids'], attention_mask=input_encoded['attention_mask'])`, in which `input_encoded` is a dictionary.

Same applies to the forward function of the model.

In [5]:
input_text = "子曰：人"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
n_outputs = 5

output = model.generate(**input_encoded, 
                    max_length=30, 
                    num_return_sequences=n_outputs,
                    do_sample=True, 
                    top_k=50, 
                    top_p=0.95, 
                    temperature=0.7,
                    pad_token_id=0,
                )
# print(type(output))
# print(output.shape)

for i in range(n_outputs):
    output_text = tokenizer.decode(output[i], skip_special_tokens=True)
    output_text = output_text.replace("\n", "")
    print(output_text)

子曰：人之为值，本乎圣人之道，仁者为本，学而知之，为贵。夫子之
子曰：人之行，如面镜子中照人，人之善，则如水镜也。这句话的意思子曰：人
子曰：人之交若玉，必有攸金。金者，德也。君子以德，以德，而治也
子曰：人之恶，吾不齿。吾有三思，以辨其善恶。三思者，思其心之
子曰：人无德，焉有国？子曰：君子有德，焉有国？子曰：君子有德，焉


We can see that the generation is far from perfect. It still has good chances to produce a lot of repetitions.

---

## T1. Implement Greedy Decoding 

Here we will practice implementing greedy decoding (top-1 sampling) manually.

Let's first pass the `input_encoded` dictionary to the `forward` function, and get the returned logits.

In [6]:
input_text = "今天天气"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)

output = model(**input_encoded)
# The above code is equivalent to:
# output = model(input_ids=input_encoded["input_ids"], 
#               attention_mask=input_encoded["attention_mask"])

print("output.keys():", output.keys())
logits = output.logits
print(logits.shape)

output.keys(): odict_keys(['logits', 'past_key_values'])
torch.Size([1, 2, 151936])


The returned `logits` tensor is of shape `(B, L, V)`, which contains the logits of all tokens. 

- `B` is batch size (here B=1);
- `L` is the length of the sequence (here L=2);
- `V` is the vocabulary size (here V=151936).

In order to conduct greedy decoding, i.e., top-1 sampling, we need to get the logits of the last token, by specifying the second dimension.

The size of the returned logits should be of the vocabulary size.

In [ ]:
### START YOUR CODE ###
# Get the probability distribution predicted at the last token's position
last_token_logits = None
### END YOUR CODE ###

# Test
print("last_token_logits.shape:", last_token_logits.shape)

# You expect to see the following output:
# last_token_logits.shape: torch.Size([151936])

last_token_logits.shape: torch.Size([151936])


Then, you simply pick the most likely token id from this vocabulary-sized distribution, by using the `argmax()` function.

In [ ]:
### START YOUR CODE ###
top1_next_id = None
### END YOUR CODE ###

# Test
print("top-1 next token id:", top1_next_id.item())

top1_next_str = tokenizer.decode(top1_next_id)
print("top-1 next token:", top1_next_str)

# You should expect to see the following output:
# top-1 next token id: 105212
# top-1 next token: 晴

top-1 next token id: 105212
top-1 next token: 晴


Next, you can all the model's `forward()` function to generate an output, with KV cache manually enabled.

- `past_key_values` is an instance of `DynamicCache`, which should be passed as an argument to the `forward()`.
- specify the argument `use_cache=True`.

In [ ]:
input_text = "今天天气"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
input_ids = input_encoded["input_ids"]
attn_mask = input_encoded["attention_mask"]

# initialize KV cache
past_key_values = DynamicCache()
print("Before forward pass\nKV cache size in # of layers:", len(past_key_values.layers))

# forward pass 
### START YOUR CODE ###
output = model(
    None
)
### END YOUR CODE ###
print()

# after forward pass, KV cache is not empty
print("After forward pass\nKV cache size in # of layers", len(past_key_values.layers))
if len(past_key_values.layers) > 0:
    print("shape of each layer:", past_key_values.layers[0].values.shape) # KV cache is of shape [batch_size, num_heads, seq_len, head_dim]

Before forward pass
KV cache size in # of layers: 0

After forward pass
KV cache size in # of layers 28
shape of each layer: torch.Size([1, 8, 2, 128])


Next, you can now implement the full generation loop with KV cache enabled, in which there is a generation loop.

- The loop continues until `max_gen_len` is reached, or a `"<endoftext>"` token is generated.
- For each loop iteration, use greedy decoding to sample a `next_id` from the output, and append it to `generated_ids`.
- Update `attention_mask` by increment it with a constant `[1]` tensor (with the correct dimension).
- Update `input_encoded` for each iteration, with `next_id` as the new `input_ids`, and the updated `attention_mask`.

In [ ]:
def generate_greedy_with_kv_cache(input_text, model, tokenizer, max_gen_len=100):
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded["input_ids"]

    # initialize KV cache
    past_key_values = DynamicCache()
    
    # to store generated token ids
    generated_ids = input_ids

    # loop until max_gen_len is reached, or a `"<endoftext>"` token is generated
    count = 0
    while count < max_gen_len:
        ### START YOUR CODE ###
        output = model(
            None
        )

        # Greedy decoding
        next_logits = None
        next_id = None
        generated_ids = None # append next_id to generated_ids

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"] 
        attention_mask = None # update attention_mask
        input_encoded = None # update input_encoded

        # break if '<|endoftext|>' is generated
        if next_id.item() == tokenizer.pad_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

In [12]:
# Test
SPECIAL_TOKEN_IDS = set([ tokenizer.pad_token_id])

input_text = "今天天气不错哟"

start_time = time.time()
generated_ids = generate_greedy_with_kv_cache(input_text, model, tokenizer, max_gen_len=100)
end_time = time.time()
print("generation speed: {:.4} tokens/s".format(generated_ids.shape[1] / (end_time - start_time)))

# Decode the generated tokens ids
print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

# You should expect to see the following output:
# generation speed: xx.xx tokens/s
# input text: 今天天气
# generated text:
# 今天天气不错哟，我们去公园散步吧。这句话中，"我们"指的是谁呢？A. 你 B. 我 C. 你和我 D. 你和我以及我
# 答案：C
# 问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和我 D. 你和我以及我
# 答案：C
# 问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和

generation speed: 28.68 tokens/s
input text: 今天天气不错哟
generated text:
今天天气不错哟，我们去公园散步吧。这句话中，"我们"指的是谁呢？A. 你 B. 我 C. 你和我 D. 你和我以及我
答案：C
问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和我 D. 你和我以及我
答案：C
问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和


Let's compare it with the `generate()` method provided by `transformers` library.

In [13]:
# Test
input_text = "今天天气不错哟"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)

start_time = time.time()
generated_ids = model.generate(**input_encoded, 
                              max_length=100, 
                              num_return_sequences=1,
                              do_sample=False,
                              use_cache=True
                            )
end_time = time.time()
print("generation speed: {:.4} tokens/s".format(generated_ids.shape[1] / (end_time - start_time)))

# Decode the generated tokens ids
print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generation speed: 31.72 tokens/s
input text: 今天天气不错哟
generated text:
今天天气不错哟，我们去公园散步吧。这句话中，"我们"指的是谁呢？A. 你 B. 我 C. 你和我 D. 你和我以及我
答案：C
问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和我 D. 你和我以及我
答案：C
问题：下面哪个选项是正确的？A. 你 B. 我 C


## T2. Implement Top-k Sampling

Now, let's implement a `top-k` sampling algorithm. We will first practice the naive idea of **uniformly** sampling from top-k most likely next tokens. 

You need to use the results returned from the PyTorch tensor built-in `topk()` method to get the top-k logits and the corresponding indices. 

In the following example, you can check the top 5 most likely words following the sentence "今天天气":

In [ ]:
k = 5
input_text = "今天天气"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)

output = model(**input_encoded)
logits = output.logits

### START YOUR CODE ###
last_token_logits = None
topk_logits, topk_indices = None
### END YOUR CODE ###

# Test
print(topk_logits)
print(topk_indices)

for i in range(k):
    tok_id = topk_indices[i].item()
    tok = tokenizer.convert_ids_to_tokens(tok_id)
    tok_str = tokenizer.convert_tokens_to_string([tok])
    print(tok_str, end=' ')

# You should expect to see the following output:
# tensor([14.5000, 14.1875, 13.8125, 13.6250, 13.1875], dtype=torch.bfloat16,
#        grad_fn=<TopkBackward0>)
# tensor([105212, 115744,  20412, 104472,  42140])
# 晴 预报 是 怎么样 多 

tensor([14.5000, 14.1875, 13.8125, 13.6250, 13.1875], dtype=torch.bfloat16,
       grad_fn=<TopkBackward0>)
tensor([105212, 115744,  20412, 104472,  42140])
晴 预报 是 怎么样 多 

Let's integrate the top-k sampling into the generation process. 

The uniform sampling can be implemented using `random.choices` among the top-k indices.

In [ ]:
def generate_topk_uniform(input_text, model, tokenizer, k=5, max_gen_len=50):
    '''
    Generate tokens from the top-k logits, and yield the sampled token id.
    Tokens are sampled from a naive uniform distribution.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded["input_ids"]

    # initialize KV cache
    past_key_values = DynamicCache()
    
    # to store generated token ids
    generated_ids = input_ids

    # loop until max_gen_len is reached, or a `"<endoftext>"` token is generated
    count = 0
    while count < max_gen_len:
        ### START YOUR CODE ###
        output = model(
            None
        )

        # Top-k sampling (with naive uniform distribution)
        next_logits = None
        topk_logits, topk_indices = None # use topk()
        next_id = None # use random.choices()
        generated_ids = None

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = None
        input_encoded = None

        # break if '<|endoftext|>' is generated
        if next_id.item() == tokenizer.pad_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

In [16]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_uniform(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟开心时光,欢迎大家来找学校玩\ 在这里看到了非常多很有品味好味道的朋友(翻译原计划是什么结构,结构后英文长音开头。 飕锅口(语为？/会怎么来中文直接入账?有没有


In [17]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_uniform(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人三曰生有 何相?
顾安远和同江一同谈及道：
安处易往不识庐！和随船同里同进事安？
家常不待宾客名将发发早。

上面句子片段


We can note that although the above uniform top-k sampling solves repetition issue, it will however produce *extremely incoherent* text. 

We should fix this by using a **proportional sampling** instead of uniform sampling.

There are plenty of different ways to implement proportionaly sampling. You can either:
- Create list of cumulative relative probabilities of the top k tokens. For instance, if the relative probabilities of $k=5$ tokens are $0.1$, $0.2$, $0.5$, $0.1$, and $0.1$, then you cumulative probability list is `cum_prob = [0.1, 0.3, 0.8, 0.9, 1.0]`. 
- Then you draw a random number $r$ from the unifrom distribution $[0,1]$ by `random.random()`, and you decide which token is sampled by telling which bin of `cum_prob` that $r$ falls into.
- Or instead, you use the `torch.multinomial()` function to accomplish similar sampling. *Note* the input weight provided to `torch.multinomial` should be the relative probabilities of the top $k$ tokens, which can be obtained from applying softmax to the logits. 

In [ ]:
def generate_topk_proportion(input_text, model, tokenizer, k=50, max_gen_len=50):
    '''
    Generate tokens from the top-k logits, and yield the sampled token id.
    Tokens are sampled proportional to their logits.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded["input_ids"]

    # initialize KV cache
    past_key_values = DynamicCache()
    
    # to store generated token ids
    generated_ids = input_ids

    # loop until max_gen_len is reached, or a `"<endoftext>"` token is generated
    count = 0
    while count < max_gen_len:
        ### START YOUR CODE ###
        output = model(
            None
        )

        # Top-k sampling (with proportional sampling)
        last_token_logits = None
        topk_logits, topk_indices = None
        topk_probs = None # use softmax

        next_id = None # Use the manual way of propotional sampling, or use torch.multinomial()
        generated_ids = None

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = None
        input_encoded = None

        # break if '<|endoftext|>' is generated
        if next_id.item() == tokenizer.pad_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

In [19]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_proportion(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟！我们去了三星级酒店，酒店前台的服务态度很棒，而且有专属早餐的安排，住宿环境舒适，性价比也很高，很适合推荐给喜欢高端酒店体验的客人。从环境、服务到价格都能觉得我们这次的入住


In [20]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_proportion(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人而不能明其性，不能知其情，必不能为天地之长。翻译成现代人通俗的话
翻译这句话的现代人通俗的解释，可以是：每个人都要知其性情，这样一个人才能长久地影响


Do you think the proportional sampling produces better text?

Have fun sampling! :)

---

## T3. Implement Top-p Sampling

Next, we will implement top-p sampling, which works in parallel to top-k sampling.

In `filter_topk_topp()`, we first filter out the logits that are not in the top-k, by setting their logit values to `-float('inf')`. 

And then filter out the logits whose cumulative probability (as computed from the altered logits from the previous step) is greater than `p`.
- You can first call `torch.sort()` to sort the logits in ascending order, and convert them to probabilities by applying `torch.softmax()`.
- Then, you can compute the cumulative probabilities by calling `torch.cumsum()`.
- Note that it is possible that the first logit alone dominates the distribution, and its cumulative probability is greater than `p`. In this case, we want to keep this logit, and remove all other logits.

In [ ]:
def filter_topk_topp(logits: torch.Tensor, k=50, p=0.9) -> torch.Tensor: 
    '''
    Filter a distribution of logits using top-k and/or nucleus (top-p) filtering
    '''
    assert logits.dim() == 1
    logits = logits.clone()

    if k > 0:
        ### START YOUR CODE ###
        topk_logits, topk_indices = None # use topk()
        logits_to_remove = None # find the indices for the logits to be removed
        logits[logits_to_remove] = -float('Inf')
        # you can use alternative ways
        ### END YOUR CODE ###
    
    if p > 0.0:
        ### START YOUR CODE ###
        logits_sorted, indices_sorted = None # use torch.sort()
        cum_probs = None # use torch.softmax() and torch.cumsum()
        indices_to_remove = # find the indices for the logits to be removed

        # It is possible that cum_probs[0] > p, in which case all logits will be removed
        # we want to avoid that, so always keep the first logit
        if indices_to_remove.shape[0] == indices_sorted.shape[0]:
            indices_to_remove = indices_sorted[1:]

        logits[indices_to_remove] = -float('Inf')
        ### END YOUR CODE ### 

    
    return logits

In [22]:
# Test filter_topk_topp
logits = torch.tensor(list(range(10))).float()
print('original logits:', logits)

logits2 = filter_topk_topp(logits, k=5, p=0.0)
print('\nk=5, p=0.0:', logits2)

logits3 = filter_topk_topp(logits, k=0, p=0.9)
print('\nk=0, p=0.9:', logits3)

logits4 = filter_topk_topp(logits, k=0, p=0.9999999)
print('\nk=0, p=0.9999999:', logits4)

logits5 = filter_topk_topp(logits, k=5, p=0.9999999)
print('\nk=5, p=0.9999999:', logits5)


# You are expected to see the following output:
# original logits: tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])
# k=5, p=0.0: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])
# k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 8., 9.])
# k=0, p=0.9999999: tensor([-inf, 1., 2., 3., 4., 5., 6., 7., 8., 9.])
# k=5, p=0.9999999: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])

original logits: tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

k=5, p=0.0: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])

k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 8., 9.])

k=0, p=0.9999999: tensor([-inf, 1., 2., 3., 4., 5., 6., 7., 8., 9.])

k=5, p=0.9999999: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])


In the following test, if all logits are `-inf`, then your top-p sampling is not correctly implemented. 

You wan to keep at least one element in the logits, whose logit value dominates the distribution. 

In [23]:
logits_special = torch.tensor(np.arange(10)**2).float()
print('original logits:', logits_special)

logits6 = filter_topk_topp(logits_special, k=0, p=0.9)
print('\nk=0, p=0.9:', logits6)


# You are expected to see the following output:
# original logits: tensor([ 0.,  1.,  4.,  9., 16., 25., 36., 49., 64., 81.])
# k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 81.])

original logits: tensor([ 0.,  1.,  4.,  9., 16., 25., 36., 49., 64., 81.])

k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 81.])


Finally, we integrate the filtering to the generation process.

Given the output of `filter_topk_topp()`, you need to use the same sampling algorithm as implemented in `generate_topk_proportion()`.

In [ ]:
def generate_topk_topp(input_text, model, tokenizer, k=50, p=0.9, max_gen_len=20):
    '''
    Generate tokens from the top-k and top-p filtered logits, and yield the sampled token id.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded.input_ids

    # initialize KV cache
    past_key_values = DynamicCache()

    # to store generated token ids
    generated_ids = input_ids

    count = 0
    while count < max_gen_len:
        output = model(
            None
        )

        # Get last token logits
        ### START YOUR CODE ###
        last_token_logits = None
        ### END YOUR CODE ###

        # Get the filtered logits by calling filter_topk_topp 
        ### START YOUR CODE ###
        filtered_logits = None
        ### END YOUR CODE ###


        # Sample from the remaining tokens in sorted_logits
        ### START YOUR CODE ###
        filtered_probs = None # use softmax
        try:
            next_id = None # use torch.multinomial() or your own version of sampling from filtered_probs
        except RuntimeError:
            raise

        generated_ids = None # append next_id to generated_ids

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = None
        input_encoded = None

        if next_id.item() == tokenizer.sep_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

Do it yourself to test if adjusting the `k` and `p` values change the generation quality.

In [25]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=100, p=0.95, max_gen_len=50)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟，真是的！因为现在的天气真是适合做很多好玩的事情，比如放风筝、做手工、赏花等，如果你愿意的话，可以一起来做一些活动。天气也暖暖的，所以相信你一定会喜欢和我一起做


In [26]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=50, p=0.9, max_gen_len=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人而不能明其志，不能明其德，不能明其志，不能明其德，不亦已乎？（《论语·述而》）

“天也，人也，天人相应。”——《汉书·唐都传》

“仁义礼智信”——《礼记·大学》

“为仁而战，以德报怨，以智为重，以信为本”——《史记·项羽本


## T4. Implement Temperature Sampling

Finally, we can adjust `filtered_logits` returned from top-k and top-p, using the temperature $\tau$, before feeding it to softmax.

In [ ]:
def generate_topk_topp(input_text, model, tokenizer, k=50, p=0.9, temperature=1.0, max_gen_len=20):
    '''
    Generate tokens from the top-k and top-p filtered logits, and yield the sampled token id.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded.input_ids

    # initialize KV cache
    past_key_values = DynamicCache()

    # to store generated token ids
    generated_ids = input_ids

    count = 0
    while count < max_gen_len:
        output = model(
            None
        )

        # Get last token logits
        ### START YOUR CODE ###
        last_token_logits = None
        ### END YOUR CODE ###

        # Get the filtered logits by calling filter_topk_topp 
        ### START YOUR CODE ###
        filtered_logits = None
        ### END YOUR CODE ###

        # Adjust the logits with temperature
        ### START YOUR CODE ###
        filtered_logits = None
        ### END YOUR CODE ###

        # Sample from the remaining tokens in sorted_logits
        ### START YOUR CODE ###
        filtered_probs = None
        try:
            next_id = None
        except RuntimeError:
            raise

        generated_ids = None

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = None
        input_encoded = None

        if next_id.item() == tokenizer.sep_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

Do it yourself to see if using higher temperature can lead to more creative generations.

In [ ]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=100, p=0.95, temperature=0.5, max_gen_len=50)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟，真是的，因为现在的天气真是让人开心。这句话的语法和结构是否正确？

这句中的“因为现在的天气真是让人开心”是主句，而“因为”是连接词。连接词后面应该跟一个完整的句子


In [35]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=50, p=0.95, temperature=0.9, max_gen_len=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人之为人都有为，为己也，不以己自任，与己为他人之道，然后可以以己为己，为己，为他。人之为人都有为，为己也，不以己自任，与己为他人之道，然后可以以己为己，为己，为他。

这段话需要重新组织，以突出个人与他人之间的相互作用，强调“他者”的角色，以及以“己”为本


Congratulations! You have completed the lab for top-k and top-p sampling.